# 5-Year Pipeline: Part 4 - Strategy Model Training (Auto-Analysis)

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("Loading 5-Year Final Data...")
df = pd.read_csv('5_year_final_data.csv')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df['date'] = pd.to_datetime(df['date'])

Loading 5-Year Final Data...


In [2]:
print("Filtering to Valid Triggered Trades...")
df_signals = df[df['base_signal'] == True]
df_signals = df_signals[~df_signals['outcome'].isin(['No_Trigger', 'Pending', 'Timeout', np.nan])]

df_signals = df_signals[df_signals['risk'] > 0]
df_signals['max_r_achieved'] = df_signals['mfe'] / df_signals['risk']
df_signals = df_signals.dropna(subset=['max_r_achieved'])
df_signals = df_signals.sort_values('date').reset_index(drop=True)

n = len(df_signals)
train_end = int(n * 0.80)
train_df = df_signals.iloc[:train_end]
test_df = df_signals.iloc[train_end:]

leakage_cols = [
    'base_signal', 'signal_strength', 'entry_price', 'stop_loss', 'risk', 'target_1_2',
    'target_hit', 'sl_hit', 'timeout', 'outcome', 'days_to_outcome',
    'return_7d_after_signal', 'max_gain_7d', 'max_loss_7d', 'mfe', 'mae', 'Trade_Triggered',
    'max_r_achieved', 'date', 'symbol', 'sector', 'Stock Symbol', 'Sector', 'Trading_Day_Of_Week'
]
base_features = [c for c in df_signals.columns if c not in leakage_cols and pd.api.types.is_numeric_dtype(df_signals[c])]
base_features = [f for f in base_features if df_signals[f].isnull().mean() < 1.0]

X_train = train_df[base_features]
y_train = train_df['max_r_achieved']
X_test = test_df[base_features]
y_test = test_df['max_r_achieved']

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_train_s = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_s = scaler.transform(imputer.transform(X_test))

Filtering to Valid Triggered Trades...


In [3]:
print("--- PHASE 1: TRAINING BASE MODEL ---")
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}
base_xgb = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
random_search = RandomizedSearchCV(base_xgb, param_distributions=param_grid, n_iter=10, cv=3, scoring='neg_mean_absolute_error', random_state=42)
random_search.fit(X_train_s, y_train)

best_base = random_search.best_estimator_
preds_base = best_base.predict(X_test_s)
mae_base = mean_absolute_error(y_test, preds_base)
print(f"Base Model MAE: {mae_base:.2f} R")

os.makedirs('models', exist_ok=True)
joblib.dump(best_base, 'models/strategy_base_model.pkl')
joblib.dump(scaler, 'models/strategy_base_scaler.pkl')
joblib.dump(imputer, 'models/strategy_base_imputer.pkl')
with open('base_features.txt', 'w') as f:
    for feature in base_features:
        f.write(f"{feature}\n")

--- PHASE 1: TRAINING BASE MODEL ---


Base Model MAE: 0.89 R


In [4]:
print("--- PHASE 2: AUTOMATIC FEATURE ANALYSIS ---")
importances = best_base.feature_importances_
importance_df = pd.DataFrame({'Feature': base_features, 'Importance': importances})

zero_imp = importance_df[importance_df['Importance'] == 0]['Feature'].tolist()

corr_matrix = pd.DataFrame(X_train_s, columns=base_features).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = []
for col in upper.columns:
    high_corr = upper[col][upper[col] > 0.90].index.tolist()
    if high_corr:
        col_imp = importance_df[importance_df['Feature'] == col]['Importance'].values[0]
        for c in high_corr:
            c_imp = importance_df[importance_df['Feature'] == c]['Importance'].values[0]
            if col_imp < c_imp:
                to_drop_corr.append(col)
                break

features_to_remove = list(set(zero_imp + to_drop_corr))
refined_features = [f for f in base_features if f not in features_to_remove]

print(f"Removed {len(features_to_remove)} bad/redundant features.")
print(f"Remaining good features: {len(refined_features)}")

X_train_ref = train_df[refined_features]
X_test_ref = test_df[refined_features]

imputer_ref = SimpleImputer(strategy='median')
scaler_ref = StandardScaler()
X_train_s_ref = scaler_ref.fit_transform(imputer_ref.fit_transform(X_train_ref))
X_test_s_ref = scaler_ref.transform(imputer_ref.transform(X_test_ref))

--- PHASE 2: AUTOMATIC FEATURE ANALYSIS ---


Removed 38 bad/redundant features.
Remaining good features: 99


In [5]:
print("--- PHASE 3: TRAINING REFINED MODEL ---")
random_search_ref = RandomizedSearchCV(base_xgb, param_distributions=param_grid, n_iter=10, cv=3, scoring='neg_mean_absolute_error', random_state=42)
random_search_ref.fit(X_train_s_ref, y_train)

best_refined = random_search_ref.best_estimator_
preds_ref = best_refined.predict(X_test_s_ref)
mae_ref = mean_absolute_error(y_test, preds_ref)
print(f"Refined Model MAE: {mae_ref:.2f} R")

joblib.dump(best_refined, 'models/5_year_xgboost_model.pkl')
joblib.dump(scaler_ref, 'models/5_year_scaler.pkl')
joblib.dump(imputer_ref, 'models/5_year_imputer.pkl')

with open('final_features.txt', 'w') as f:
    for feature in refined_features:
        f.write(f"{feature}\n")

print("Saved models successfully! Notebook 5 will evaluate and log results.")

--- PHASE 3: TRAINING REFINED MODEL ---


Refined Model MAE: 0.89 R
Saved models successfully! Notebook 5 will evaluate and log results.
